# 🎬 Vecna / AIC51 — Pipeline chạy lại M09 & M10 trên Colab Pro+

Notebook này chạy trọn pipeline cho 2 batch video **M09** và **M10**:
tải video → `aic51-cli add` (keyframe + thumbnail + audio + video_info) → 4 feature
(**Qwen-VL → SigLIP 2 → OCR → ASR**) → đối soát → nén **2 file ZIP** lên Google Drive.

**Cách dùng**
1. `Runtime → Change runtime type` → **A100** (hoặc L4) + **High-RAM**.
2. Sửa cấu hình ở **Cell 1** nếu cần (thư mục Drive, backend OCR…).
3. `Runtime → Run all`. Chỉ cần bấm cho phép khi Colab hỏi quyền **Google Drive** (ngay đầu).

**Những điểm đã điều chỉnh so với prompt gốc cho khớp code thật của repo (`aic51-src`)**

| Prompt gốc | Thực tế trong repo | Notebook làm gì |
|---|---|---|
| `aic51-cli analyse -m <feature>` | `analyse` **không có** `-m`; chọn model bằng `--use-qwen-vl`, `--use-image-siglip`, `--use-ocr`, `--use-asr` (với `add`, `-m` lại là `--move`) | Dùng đúng các cờ `--use-*`, kèm `--keep-going` |
| Config của repo (`ocr.source: tesseract`) | Dùng **FINALCONFIG** (`ocr.source: paddle_vietocr`, `vgg_transformer`, batch 16) | Config được **nhúng nguyên văn** ở Cell 3 và ghi thẳng làm `workspace/config.yaml` |
| PaddleOCR / VietOCR bản mới nhất | Code repo gọi API **PaddleOCR 2.x** (`show_log`, `use_gpu`, `ocr(rec=False, cls=False)`) — PaddleOCR 3.x đã bỏ các tham số này | Cài `paddleocr==2.10.0` + `paddlepaddle` (CPU, vì repo chạy detect bằng CPU) + `vietocr`; VietOCR nhận dạng trên GPU |
| — | `vietocr` tải file config từ `vocr.vn`, không có đường dự phòng | Vá `download_config`: thử `vocr.vn` → mirror GitHub `pbcquoc/vietocr`, có cache; weights VietOCR được cache lên Drive cho lần sau |
| Xoá video ngay sau `add` | ASR (`asr.py`) đọc FPS từ `data/videos/<id>.mp4`; thiếu file → ASR lỗi | `add` chạy với `-m` (move, không nhân đôi dung lượng) → `raw_videos` trống ngay; `data/videos` được xoá **sau khi ASR xong** |
| Extractor thiếu thư viện | Repo **bỏ qua im lặng** extractor nào import lỗi (`invalid feature extractor`) | Cell *Preflight* chạy thử cả 4 model trên ảnh/âm thanh giả trước khi xử lý dữ liệu thật |

Mọi bước đều in `[YYYY-MM-DD HH:MM:SS]` bắt đầu/kết thúc, `Elapsed time`, RAM / đĩa / VRAM trước–sau,
và được ghi song song vào file log (copy lên Drive ở cuối).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — CẤU HÌNH (chỉ cần sửa cell này)
# ════════════════════════════════════════════════════════════════════════════
REPO_URL    = "https://github.com/JimmyK300/Vecna.git"
REPO_BRANCH = "main"

REPO_DIR      = "/content/Vecna"
WORKSPACE     = "/content/workspace"        # workspace aic51 (ổ SSD local của Colab)
DOWNLOAD_DIR  = "/content/downloads"        # nơi aria2c lưu file .zip
RAW_VIDEO_DIR = "/content/raw_videos"       # video thô sau giải nén (được làm phẳng)
EXPORT_DIR    = "/content/export"           # nơi tạo ZIP trước khi copy lên Drive
LOG_DIR       = "/content/pipeline_logs"

DRIVE_MOUNT      = "/content/drive"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AIC2026/Vecna_M09_M10"

BATCHES = {
    "M09": "https://aic-data.ledo.io.vn/Videos_M09.zip",
    "M10": "https://aic-data.ledo.io.vn/Videos_M10.zip",
}

# Thứ tự trích xuất: (tên feature trong config.yaml, cờ CLI của `aic51-cli analyse`)
FEATURE_ORDER = [
    ("qwen_vl",                  "--use-qwen-vl"),
    ("image_siglip2_so400m-378", "--use-image-siglip"),
    ("ocr",                      "--use-ocr"),
    ("asr",                      "--use-asr"),
]

FEATURES_ZIP_NAME   = "features_M09_M10.zip"
THUMBNAILS_ZIP_NAME = "thumbnails_M09_M10.zip"

# Config dùng cho workspace: để trống = dùng FINALCONFIG nhúng ở Cell 3.
# Hoặc trỏ tới 1 file trên Drive, ví dụ "/content/drive/MyDrive/AIC2026/FINALCONFIG.yaml"
CONFIG_FILE_OVERRIDE = ""

# Cache weights VietOCR trên Drive (vocr.vn thỉnh thoảng sập): có file → dùng luôn; chưa có → lưu sau preflight
VIETOCR_WEIGHTS_CACHE = "/content/drive/MyDrive/AIC2026/_cache/vgg_transformer.pth"

AUTO_TUNE_BATCH      = False  # True = tăng batch_size Qwen-VL (16/8) & SigLIP (128/64) theo VRAM để chạy nhanh hơn
MAX_FEATURE_ATTEMPTS = 3      # mỗi feature: nếu còn frame thiếu → chạy lại (tự giảm 1/2 batch)
FILL_EMPTY_ASR_FOR_SILENT_VIDEOS = True  # video không có track audio → asr.npy = "" (giống ASR không nghe được gì)
DELETE_VIDEOS_AFTER_ASR = True           # xoá data/videos sau khi ASR hoàn tất
DELETE_KEYFRAMES_BEFORE_ZIP = False      # True = xoá data/keyframes trước khi nén (tự bật nếu thiếu đĩa)
STRICT_VERIFY      = True     # đối soát lỗi → dừng, KHÔNG nén/upload dữ liệu thiếu
COPY_LOGS_TO_DRIVE = True     # copy log + báo cáo đối soát vào DRIVE_OUTPUT_DIR/logs/
FLUSH_DRIVE_AT_END = True     # flush_and_unmount Drive ở cuối để chắc chắn upload xong

LOG_TZ = "Asia/Ho_Chi_Minh"   # múi giờ in log
DISK_SAFETY_MARGIN_GB = 25    # chừa trống cho keyframe/audio/model cache
HEARTBEAT_SEC = 300           # lệnh chạy lâu: cứ 5 phút in tiến độ + RAM/đĩa/VRAM

print("✅ Đã nạp cấu hình.", flush=True)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2 — TIỆN ÍCH LOGGING: timestamp, đo thời gian, RAM/đĩa/VRAM, chạy lệnh
# ════════════════════════════════════════════════════════════════════════════
import contextlib, datetime as _dt, json, os, re, shlex, shutil, subprocess, sys, threading, time
from collections import deque
from pathlib import Path
from zoneinfo import ZoneInfo

try:
    import psutil
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil"], check=True)
    import psutil

_TZ = ZoneInfo(LOG_TZ)
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)
LOG_FILE = Path(LOG_DIR) / f"pipeline_{_dt.datetime.now(_TZ):%Y%m%d_%H%M%S}.log"
_LOG_LOCK = threading.Lock()
PIPELINE_START = time.time()
STEP_RESULTS = []


def now_str():
    return _dt.datetime.now(_TZ).strftime("%Y-%m-%d %H:%M:%S")


def log(msg=""):
    """In ra màn hình (flush=True) và ghi đồng thời vào LOG_FILE."""
    line = f"[{now_str()}] {msg}"
    with _LOG_LOCK:
        print(line, flush=True)
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line + "\n")


def fmt_bytes(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024 or unit == "TB":
            return f"{n:.1f} {unit}" if unit != "B" else f"{int(n)} B"
        n /= 1024


def fmt_elapsed(sec):
    m, s = divmod(int(round(sec)), 60)
    return f"{m} phút {s} giây"


def gpu_status():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception:
        return "n/a (không có nvidia-smi)"
    parts = []
    for row in filter(None, out.splitlines()):
        idx, name, used, total, util = [x.strip() for x in row.split(",")]
        parts.append(f"GPU{idx} {name}: {used}/{total} MiB (util {util}%)")
    return " | ".join(parts) or "n/a"


def sys_stats(tag):
    vm = psutil.virtual_memory()
    du = shutil.disk_usage("/content" if os.path.isdir("/content") else "/")
    log(f"   [{tag}] RAM used {fmt_bytes(vm.used)}/{fmt_bytes(vm.total)} "
        f"(available {fmt_bytes(vm.available)}) | Disk free {fmt_bytes(du.free)}/{fmt_bytes(du.total)} "
        f"| VRAM {gpu_status()}")


@contextlib.contextmanager
def step(name, heavy=True):
    """Bọc một bước: timestamp bắt đầu/kết thúc, Elapsed time, tài nguyên trước/sau."""
    t0 = time.time()
    start = now_str()
    log("=" * 100)
    log(f"▶ BẮT ĐẦU: {name}")
    log(f"   Start: [{start}]")
    if heavy:
        sys_stats("TRƯỚC")
    ok = False
    try:
        yield
        ok = True
    finally:
        el = time.time() - t0
        if heavy:
            sys_stats("SAU  ")
        log(f"{'✅ HOÀN THÀNH' if ok else '❌ LỖI'}: {name}")
        log(f"   End:   [{now_str()}]")
        log(f"   Elapsed time: {fmt_elapsed(el)}")
        STEP_RESULTS.append({"step": name, "ok": ok, "start": start, "end": now_str(), "seconds": round(el, 1)})


def build_env(extra=None):
    env = os.environ.copy()
    env.update({
        "PYTHONUNBUFFERED": "1",
        "PYTHONIOENCODING": "utf-8",
        "COLUMNS": "160",
        "TOKENIZERS_PARALLELISM": "false",
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    })
    env.update(globals().get("EXTRA_ENV", {}))
    if extra:
        env.update(extra)
    return env


_PROGRESS_RE = re.compile(r"\d+%\||it/s\]|s/it\]|^Progress: |\[#\w+ ")


def run_cmd(cmd, cwd=None, check=True, env=None, heartbeat=None, progress_every=30):
    """Chạy lệnh, stream log từng dòng (flush), gom dòng progress bar, heartbeat định kỳ."""
    shell = isinstance(cmd, str)
    shown = cmd if shell else " ".join(shlex.quote(str(c)) for c in cmd)
    log(f"$ {shown}" + (f"   (cwd={cwd})" if cwd else ""))
    proc = subprocess.Popen(
        cmd if shell else [str(c) for c in cmd], cwd=cwd, env=env or build_env(), shell=shell,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace")
    t0 = time.time()
    stop = threading.Event()

    def _beat():
        while not stop.wait(HEARTBEAT_SEC):
            extra = ""
            if heartbeat:
                try:
                    extra = " | " + str(heartbeat())
                except Exception as e:  # heartbeat không được làm hỏng lệnh chính
                    extra = f" | heartbeat lỗi: {e}"
            log(f"   ⏳ vẫn đang chạy ({fmt_elapsed(time.time() - t0)}){extra}")
            sys_stats("heartbeat")

    beat = threading.Thread(target=_beat, daemon=True)
    beat.start()
    tail = deque(maxlen=80)
    last_prog_t, last_prog = 0.0, None
    try:
        for raw in proc.stdout:   # text mode: '\r' của progress bar cũng được tách dòng
            line = raw.rstrip()
            if not line.strip():
                continue
            tail.append(line)
            if _PROGRESS_RE.search(line):
                last_prog = line
                if time.time() - last_prog_t >= progress_every:
                    log("   " + line[-300:])
                    last_prog_t = time.time()
                continue
            log("   " + line)
        rc = proc.wait()
    finally:
        stop.set()
    if last_prog:
        log("   " + last_prog[-300:])
    if check and rc != 0:
        raise RuntimeError(f"Lệnh thất bại (exit={rc}): {shown}\n--- 80 dòng cuối ---\n" + "\n".join(tail))
    return rc


def dir_size(path):
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            try:
                total += os.path.getsize(os.path.join(root, fn))
            except OSError:
                pass
    return total


def disk_free():
    return shutil.disk_usage("/content" if os.path.isdir("/content") else "/").free


log(f"Logging sẵn sàng. File log: {LOG_FILE}")
sys_stats("KHỞI ĐỘNG")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 3 — FINALCONFIG (nhúng nguyên văn → ghi thành workspace/config.yaml ở Cell 7)
# ════════════════════════════════════════════════════════════════════════════
import yaml

FINAL_CONFIG_YAML = """# max workers ratio defining how much CPU cores to use
max_workers_ratio: 1.0

add:
  default_size: [1280, 720]
  # Maximum length between two consecutive keyframes (in seconds)
  max_scene_length: 2
  # Keyframe size ratio to the original frame. This also affects resolution of videos
  keyframe_resize_ratio: 1.0
  # Thumbnail size ratio to the keyframe
  thumbnail_resize_ratio: 0.25
  # Max clips length around the keyframes (in seconds)
  clip_length: 7
  # Video compress ratio
  compress_size_rate: 0.5

analyse:
  # num_workers of DataLoader
  num_workers: 4
  # pin_memory of DataLoader
  pin_memory: true

milvus:
  # Extra fields (apart from features)
  fields:
    - field_name: "frame_id"
      datatype: "VARCHAR"
      max_length: 32
      is_primary: true

# List of features
features: &analyse_features
  # image_clip_pe-l-14-336:
  #   model: "image_clip"
  #   source: "open_clip"
  #   arch_name: "PE-Core-L-14-336"
  #   pretrained_model: "meta"
  #   analyse:
  #     batch_size: 64
  #   index:
  #     datatype: "FLOAT_VECTOR"
  #     dim: 1024
  #     metric_type: "COSINE"
  #     index_type: "SCANN"
  #     params:
  #       nlist: 512

  image_siglip2_so400m-378:
    model: "image_siglip"
    source: "open_clip"
    arch_name: "ViT-SO400M-14-SigLIP2-378"
    pretrained_model: "webli"
    analyse:
      batch_size: 32
    index:
      datatype: "FLOAT_VECTOR"
      dim: 1152
      metric_type: "COSINE"
      index_type: "SCANN"
      params:
        nlist: 512

  qwen_vl:
    model: "qwen_vl_embedding"
    pretrained_model: "Qwen/Qwen3-VL-Embedding-2B"
    analyse:
      batch_size: 1
    index:
      datatype: "FLOAT_VECTOR"
      dim: 2048
      metric_type: "COSINE"
      index_type: "SCANN"
      params:
        nlist: 512

  ocr:
    model: "ocr"
    source: "paddle_vietocr"
    pretrained_model: "vgg_transformer"
    analyse:
      batch_size: 16
      pad_y: 6
      pad_x: 4
      det_lang: "vi"
    index:
      default_value: ""
      datatype: "VARCHAR"
      max_length: 8192
      index_type: "BM25"
      params:
        bm25_k1: 1.2
        bm25_b: 0.75
        inverted_index_algo: "TAAT_NAIVE"

  asr:
    model: "asr"
    source: "whisperx"
    arch_name: "large-v3-turbo"
    analyse:
      batch_size: 16
    index:
      default_value: ""
      datatype: "VARCHAR"
      max_length: 8192
      index_type: "BM25"
      params:
        bm25_k1: 1.2
        bm25_b: 0.75
        inverted_index_algo: "TAAT_NAIVE"

searcher:
  language_models:
    # language_clip_pe-l-14-336:
    #   model: "image_clip"
    #   source: "open_clip"
    #   arch_name: "PE-Core-L-14-336"
    #   pretrained_model: "meta"
    #   target:
    #     - image_clip_pe-l-14-336

    language_siglip2_so400m-378:
      model: "image_siglip"
      source: "open_clip"
      arch_name: "ViT-SO400M-14-SigLIP2-378"
      pretrained_model: "webli"
      target:
        - image_siglip2_so400m-378

    language_qwen_vl:
      model: "qwen_vl_embedding"
      pretrained_model: "Qwen/Qwen3-VL-Embedding-2B"
      target:
        - qwen_vl

  ocr:
    ocr_field: "ocr_sparse"

  asr:
    asr_field: "asr_sparse"

frontend:
  dev_port: 5173

backends:
  core:
    port: 6900
    search_proxy:
      request_timeout: null
      max_concurrent_requests: 10
      servers:
        - host: http://127.0.0.1:1337
    file_proxy:
      request_timeout: null
      max_concurrent_requests: 10
      servers:
        - host: http://127.0.0.1:4200

  search:
    port: 1337
    collection: "workspace_col"
    workers: 1
    gpu: true

  file:
    port: 4200
    workers: 1
"""

if CONFIG_FILE_OVERRIDE:
    FINAL_CONFIG_TEXT = Path(CONFIG_FILE_OVERRIDE).read_text(encoding="utf-8")
    CONFIG_SOURCE_DESC = CONFIG_FILE_OVERRIDE
else:
    FINAL_CONFIG_TEXT = FINAL_CONFIG_YAML.lstrip("\n")
    CONFIG_SOURCE_DESC = "FINALCONFIG nhúng trong notebook"
_final = yaml.safe_load(FINAL_CONFIG_TEXT)
for _f, _ in FEATURE_ORDER:
    if _f not in (_final.get("features") or {}):
        raise RuntimeError(f"Config thiếu feature '{_f}'")
OCR_SOURCE = str(_final["features"]["ocr"].get("source", "")).lower()
log(f"Config: {CONFIG_SOURCE_DESC}")
for _f, _ in FEATURE_ORDER:
    _c = _final["features"][_f]
    log(f"   {_f:28s} model={_c.get('model')} source={_c.get('source')} "
        f"arch={_c.get('arch_name')} pretrained={_c.get('pretrained_model')} batch={(_c.get('analyse') or {}).get('batch_size')}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 — KIỂM TRA GPU & MOUNT GOOGLE DRIVE (làm ngay đầu để không phải chờ xác thực giữa chừng)
# ════════════════════════════════════════════════════════════════════════════
with step("Kiểm tra GPU & mount Google Drive", heavy=False):
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                       capture_output=True, text=True)
    if q.returncode != 0 or not q.stdout.strip():
        raise RuntimeError("Không thấy GPU. Vào Runtime → Change runtime type → chọn A100 hoặc L4.")
    GPU_NAME, GPU_MEM_MB = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
    GPU_MEM_MB = int(float(GPU_MEM_MB))
    log(f"GPU: {GPU_NAME} ({GPU_MEM_MB} MiB) | CPU: {os.cpu_count()} vCPU | "
        f"RAM: {fmt_bytes(psutil.virtual_memory().total)} | Python {sys.version.split()[0]}")
    if GPU_MEM_MB < 20000:
        log("⚠️  VRAM < 20 GB: Qwen3-VL-Embedding-2B vẫn chạy được với batch mặc định nhưng sẽ chậm.")

    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    log(f"Drive OK → {DRIVE_OUTPUT_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 5 — GÓI HỆ THỐNG: aria2, ffmpeg, unzip/zip (+ tesseract nếu config dùng tesseract)
# ════════════════════════════════════════════════════════════════════════════
with step("Cài gói hệ thống (apt)"):
    NEED_TESSERACT = OCR_SOURCE == "tesseract"
    run_cmd("apt-get update -qq", check=False)
    run_cmd("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq aria2 ffmpeg unzip zip"
            + (" tesseract-ocr tesseract-ocr-eng tesseract-ocr-vie" if NEED_TESSERACT else "") + " > /dev/null")
    for tool in ("aria2c", "ffmpeg", "ffprobe", "unzip", "zip") + (("tesseract",) if NEED_TESSERACT else ()):
        if not shutil.which(tool):
            raise RuntimeError(f"Thiếu công cụ hệ thống: {tool}")
    if NEED_TESSERACT:
        langs = subprocess.run(["tesseract", "--list-langs"], capture_output=True, text=True).stdout.split()
        if not {"eng", "vie"} <= set(langs):
            raise RuntimeError(f"Tesseract thiếu gói ngôn ngữ eng/vie: {langs}")
    log("aria2c, ffmpeg, ffprobe, unzip, zip" + (", tesseract(eng+vie)" if NEED_TESSERACT else "") + ": OK")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — CLONE REPO & CÀI PYTHON DEPENDENCIES
#  • Khoá torch/torchvision/torchaudio/numpy về đúng bản Colab đang có (constraints) để pip không
#    kéo torch khác bản CUDA → không cần restart runtime.
#  • `pip install -e .` với --no-deps: pyproject ghim numpy==1.26.4 sẽ phá môi trường Colab (numpy 2.x);
#    các dependency được cài tường minh bên dưới.
#  • whisperx cài --no-deps (whisperx ghim torch~=2.8), đúng như note.md của repo.
# ════════════════════════════════════════════════════════════════════════════
from importlib import metadata as _md

with step("Clone repo Vecna & cài Python dependencies"):
    if Path(REPO_DIR, ".git").exists():
        run_cmd(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH])
        run_cmd(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"])
    else:
        run_cmd(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR])
    REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                 capture_output=True, text=True).stdout.strip()
    log(f"Repo commit: {REPO_COMMIT}")

    pins = []
    for pkg in ("torch", "torchvision", "torchaudio", "torchcodec", "numpy"):
        try:
            pins.append(f"{pkg}=={_md.version(pkg)}")
        except _md.PackageNotFoundError:
            pass
    # Giữ mọi biến thể opencv cùng 1 phiên bản (paddleocr kéo opencv-contrib-python; lệch bản → cv2 hỏng)
    _cv = None
    for pkg in ("opencv-python", "opencv-python-headless", "opencv-contrib-python"):
        try:
            _cv = _cv or _md.version(pkg)
        except _md.PackageNotFoundError:
            pass
    if _cv:
        pins += [f"{pkg}=={_cv}" for pkg in ("opencv-python", "opencv-python-headless", "opencv-contrib-python")]
    if not any(p.startswith("torchcodec") for p in pins):
        # torchcodec phải khớp bản torch (pyannote.audio 4 cần torchcodec)
        _tv = _md.version("torch").split("+")[0]
        _tc = {"2.8": "0.7.*", "2.9": "0.8.*"}.get(".".join(_tv.split(".")[:2]))
        if _tc:
            pins.append(f"torchcodec=={_tc}")
    CONSTRAINTS = Path(LOG_DIR) / "pip_constraints.txt"
    CONSTRAINTS.write_text("\n".join(pins) + "\n")
    log("Constraints: " + ", ".join(pins))

    PIP = [sys.executable, "-m", "pip", "install", "-q", "--no-input", "-c", str(CONSTRAINTS)]

    # 1) CLI nội bộ (editable)
    run_cmd(PIP + ["--no-deps", "-e", "."], cwd=f"{REPO_DIR}/aic51-src")
    # 2) Dependency lõi của aic51 + Qwen-VL + OpenCLIP/SigLIP 2
    run_cmd(PIP + [
        "rich", "python-dotenv", "pyyaml", "pillow", "requests", "sentencepiece", "omegaconf",
        "pymilvus>=2.6.0", "fastapi", "uvicorn", "apscheduler", "deep-translator", "pytesseract",
        "transformers>=4.57.0", "accelerate", "qwen-vl-utils>=0.0.14",
        "sentence-transformers>=5.4,<6",          # 5.4 = bản đầu có SentenceTransformer đa phương thức
        "open_clip_torch>=2.31", "timm>=1.0.15",  # SigLIP 2 (ViT-SO400M-14-SigLIP2-378)
    ])
    try:
        import cv2  # noqa: F401  (Colab có sẵn opencv-python)
    except ImportError:
        run_cmd(PIP + ["opencv-python-headless"])
    # 3) ASR: faster-whisper + ctranslate2 + pyannote.audio, rồi whisperx --no-deps
    run_cmd(PIP + ["faster-whisper>=1.2.0", "ctranslate2>=4.5.0",
                   "nltk", "pandas", "omegaconf"])
    run_cmd(PIP + ["--no-deps", "whisperx"])
    # Vá whisperx: tạo stub pyannote + chuyển VAD mặc định sang Silero (không cần token HF)
    import site
    for _sp in site.getsitepackages():
        _pyd = Path(_sp) / "pyannote"
        (_pyd / "audio" / "core").mkdir(parents=True, exist_ok=True)
        (_pyd / "audio" / "pipelines").mkdir(parents=True, exist_ok=True)
        (_pyd / "__init__.py").write_text("__version__ = '4.0.0'\n")
        (_pyd / "audio" / "__init__.py").write_text("class Model: pass\nclass Pipeline: pass\nclass Inference: pass\n")
        (_pyd / "audio" / "core" / "__init__.py").write_text("")
        (_pyd / "audio" / "core" / "io.py").write_text("class AudioFile: pass\n")
        (_pyd / "audio" / "pipelines" / "__init__.py").write_text("class VoiceActivityDetection: pass\n")
        (_pyd / "audio" / "pipelines" / "utils.py").write_text("class PipelineModel: pass\n")

    _wx_asr = Path(subprocess.run([sys.executable, "-c", "import whisperx; from pathlib import Path; print(Path(whisperx.__file__).parent / 'asr.py')"],
                                  capture_output=True, text=True, check=True).stdout.strip())
    if _wx_asr.exists():
        _wxt = _wx_asr.read_text(encoding="utf-8")
        _wxt = _wxt.replace('vad_method: Optional[str] = "pyannote"', 'vad_method: Optional[str] = "silero"')
        _wx_asr.write_text(_wxt, encoding="utf-8")

    _repo_asr = Path(REPO_DIR) / "aic51-src/aic51/packages/analyse/features/asr.py"
    if _repo_asr.exists():
        _rat = _repo_asr.read_text(encoding="utf-8")
        if 'vad_method="silero"' not in _rat:
            _rat = _rat.replace('compute_type=compute_type,', 'compute_type=compute_type,\n            vad_method="silero",')
            _repo_asr.write_text(_rat, encoding="utf-8")
    # 4) OCR PaddleOCR 2.x (detect, CPU) + VietOCR (recognize, GPU) — code repo dùng API PaddleOCR 2.x
    if OCR_SOURCE != "tesseract":
        # paddle CPU là đủ: repo khởi tạo PaddleOCR(use_gpu=False). Thử 3.0.0 (đã kiểm chứng với
        # paddleocr 2.10), nếu Python của Colab quá mới thì lấy bản 3.x mới nhất.
        if run_cmd(PIP + ["paddlepaddle==3.0.0"], check=False) != 0:
            run_cmd(PIP + ["paddlepaddle>=3.0,<4"])
        run_cmd(PIP + ["paddleocr==2.10.0"])
        run_cmd(PIP + ["--no-deps", "vietocr==0.3.13"])   # vietocr ghim pillow==10.2, einops==0.2 → cài --no-deps
        run_cmd(PIP + ["einops", "gdown", "prefetch_generator", "lmdb"])
        VIETOCR_CONFIG_PATCH = """

# VECNA_PATCH_DOWNLOAD_CONFIG
def download_config(id):
    import requests, yaml
    urls = [
        f"https://vocr.vn/data/vietocr/config/{id}",
        f"https://raw.githubusercontent.com/pbcquoc/vietocr/master/config/{id}",
    ]
    for url in urls:
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200 and r.text.strip():
                return yaml.safe_load(r.text)
        except Exception:
            continue
    raise RuntimeError(f"Cannot download VietOCR config for {id}")
"""
        # Vá vietocr.tool.utils.download_config: vocr.vn → mirror GitHub, có timeout + cache
        _vu = Path(subprocess.run([sys.executable, "-c", "import vietocr.tool.utils as u; print(u.__file__)"],
                                  capture_output=True, text=True, check=True).stdout.strip())
        _src = _vu.read_text(encoding="utf-8")
        if "VECNA_PATCH_DOWNLOAD_CONFIG" not in _src:
            _vu.write_text(_src + VIETOCR_CONFIG_PATCH, encoding="utf-8")
            log(f"Đã vá {_vu} (download_config có fallback GitHub).")
    run_cmd([sys.executable, "-m", "pip", "check"], check=False)

    # Thư viện CUDA (cuDNN 9, cuBLAS) từ wheel nvidia-* cho ctranslate2
    import site
    _nv = sorted({str(p) for sp in site.getsitepackages() for p in Path(sp).glob("nvidia/*/lib")})
    EXTRA_ENV = {"LD_LIBRARY_PATH": ":".join(_nv + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(":")}
    # HF token (tuỳ chọn) từ Colab Secrets → tải model nhanh/ổn định hơn
    try:
        from google.colab import userdata
        _tok = userdata.get("HF_TOKEN")
        if _tok:
            EXTRA_ENV["HF_TOKEN"] = _tok
            log("Dùng HF_TOKEN từ Colab Secrets.")
    except Exception:
        pass

    if not shutil.which("aic51-cli"):
        raise RuntimeError("Không tìm thấy lệnh aic51-cli sau khi cài đặt.")
    run_cmd(["aic51-cli", "--help"])
    for pkg in ("torch", "numpy", "transformers", "sentence-transformers", "open_clip_torch",
                "whisperx", "faster-whisper", "ctranslate2", "pyannote.audio",
                "paddlepaddle", "paddleocr", "vietocr", "opencv-python", "opencv-contrib-python"):
        try:
            log(f"   {pkg}=={_md.version(pkg)}")
        except _md.PackageNotFoundError:
            log(f"   {pkg}: CHƯA CÀI")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 — KHỞI TẠO WORKSPACE (aic51-cli init) + GHI FINALCONFIG THÀNH config.yaml
# ════════════════════════════════════════════════════════════════════════════
import yaml

CONFIG_PATH = Path(WORKSPACE) / "config.yaml"


def load_cfg():
    with open(CONFIG_PATH, encoding="utf-8") as f:
        return yaml.safe_load(f)


def save_cfg(cfg):
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)


with step("Khởi tạo workspace & config.yaml", heavy=False):
    Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
    run_cmd(["aic51-cli", "init"], cwd=WORKSPACE)
    shutil.copy(Path(REPO_DIR) / "config.yaml", Path(WORKSPACE) / "config.repo.yaml")  # chỉ để đối chiếu
    CONFIG_PATH.write_text(FINAL_CONFIG_TEXT, encoding="utf-8")   # FINALCONFIG nguyên văn (giữ comment)
    log(f"config.yaml ← {CONFIG_SOURCE_DESC}")
    cfg = load_cfg()
    changed = False

    for fname, _ in FEATURE_ORDER:
        if fname not in (cfg.get("features") or {}):
            raise RuntimeError(f"Feature '{fname}' không có trong config.yaml của repo")
    extra_feats = [k for k in cfg["features"] if k not in dict(FEATURE_ORDER)]
    if extra_feats:
        log(f"⚠️  config còn feature khác (không chạy trong notebook này): {extra_feats}")

    # Không để lộ API key trong workspace
    if ((cfg.get("searcher") or {}).get("llm") or {}).get("api_key"):
        cfg["searcher"]["llm"]["api_key"] = ""
        changed = True
        log("Đã xoá searcher.llm.api_key trong bản config của workspace.")

    log(f"OCR source: {cfg['features']['ocr']['source']} | pretrained_model: {cfg['features']['ocr'].get('pretrained_model')}")
    if AUTO_TUNE_BATCH:
        # Keyframe cùng kích thước 1280x720 → đổi batch không đổi kết quả embedding, chỉ nhanh hơn
        tuned = {}
        if GPU_MEM_MB >= 38000:          # A100 40/80 GB
            tuned = {"qwen_vl": 16, "image_siglip2_so400m-378": 128}
        elif GPU_MEM_MB >= 22000:        # L4 24 GB
            tuned = {"qwen_vl": 8, "image_siglip2_so400m-378": 64}
        for k, v in tuned.items():
            cfg["features"][k].setdefault("analyse", {})["batch_size"] = v
            changed = True
    if changed:
        save_cfg(cfg)
    for fname, _ in FEATURE_ORDER:
        log(f"   {fname:28s} batch_size = {cfg['features'][fname].get('analyse', {}).get('batch_size')}")
    log(f"   analyse.num_workers = {cfg['analyse']['num_workers']}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 8 — PREFLIGHT: chạy thử 4 extractor đúng code của repo trên ảnh/âm thanh giả
#  (repo bỏ qua im lặng extractor import lỗi → phải bắt lỗi TRƯỚC khi xử lý hàng giờ dữ liệu).
#  Bước này cũng tải sẵn model vào cache HuggingFace.
# ════════════════════════════════════════════════════════════════════════════
PREFLIGHT_PY = Path(LOG_DIR) / "preflight.py"
PREFLIGHT_PY.write_text(r"""
import gc, importlib, shutil, sys, time, traceback
from pathlib import Path
import numpy as np
import torch
from aic51.packages.config import GlobalConfig

WORK = Path.cwd()
GlobalConfig.set_work_dir(WORK)
from aic51.packages.analyse.features import FeatureExtractorFactory

print(f"torch {torch.__version__} | cuda={torch.cuda.is_available()} | cuda_ver={torch.version.cuda}", flush=True)
if not torch.cuda.is_available():
    sys.exit("CUDA không khả dụng trong subprocess")
device = torch.device("cuda")
FEATURES = sys.argv[1].split(",")
MODULE_OF = {"qwen_vl_embedding": "qwen_vl", "image_siglip": "image_siglip", "ocr": "ocr", "asr": "asr"}

bad = []
for feat in FEATURES:
    model = GlobalConfig.get("features", feat, "model")
    if FeatureExtractorFactory.get(model) is None:
        try:
            importlib.import_module(f"aic51.packages.analyse.features.{MODULE_OF.get(model, model)}")
            bad.append(f"{feat}: model '{model}' không được đăng ký")
        except Exception:
            bad.append(f"{feat}: import lỗi\n{traceback.format_exc()}")
if bad:
    print("\n".join(bad), flush=True)
    sys.exit(2)

from PIL import Image, ImageDraw, ImageFont
tmp = WORK / ".preflight" / "keyframes"
tmp.mkdir(parents=True, exist_ok=True)
img = Image.new("RGB", (1280, 720), (20, 40, 90))
draw = ImageDraw.Draw(img)
big_font = True
try:
    font = ImageFont.truetype("DejaVuSans-Bold.ttf", 72)
except Exception:
    font, big_font = ImageFont.load_default(), False
draw.text((80, 300), "THỜI SỰ 19H - HÀ NỘI", fill="white", font=font)
img_path = tmp / "000000.jpg"
img.save(img_path, quality=90)

def init_kwargs(feat):
    g = lambda *k: GlobalConfig.get("features", feat, *k)
    return dict(source=g("source"), arch_name=g("arch_name"), pretrained_model=g("pretrained_model"),
                name=feat, batch_size=g("analyse", "batch_size") or 1, device=device, work_dir=WORK)

for feat in FEATURES:
    t0 = time.time()
    model = GlobalConfig.get("features", feat, "model")
    print(f"--- {feat} ({model}) ---", flush=True)
    ext = FeatureExtractorFactory.get(model).from_pretrained(**init_kwargs(feat))
    if model == "asr":
        import ctranslate2
        n = ctranslate2.get_cuda_device_count()
        print(f"ctranslate2 {ctranslate2.__version__} cuda devices={n}", flush=True)
        assert n > 0, "ctranslate2 không thấy GPU"
        sr = 16000
        audio = (0.1 * np.sin(2 * np.pi * 440 * np.arange(sr * 5) / sr)).astype(np.float32)
        res = ext._model.transcribe(audio, batch_size=4)
        print(f"whisperx transcribe OK, segments={len(res.get('segments', []))}", flush=True)
    else:
        out = np.asarray(ext.get_features([img_path]))
        print(f"output shape={out.shape} dtype={out.dtype}", flush=True)
        dim = GlobalConfig.get("features", feat, "index", "dim")
        if dim:
            assert out.shape == (1, dim), f"sai shape: {out.shape} != (1, {dim})"
            norm = float(np.linalg.norm(out[0]))
            assert abs(norm - 1) < 1e-2, f"vector chưa chuẩn hoá: norm={norm}"
        else:
            text = str(out[0])
            print(f"OCR text: {text[:120]!r}", flush=True)
            if big_font and not text.strip():
                sys.exit("OCR trả về chuỗi rỗng trên ảnh có chữ to → pipeline OCR không hoạt động đúng")
    del ext
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[OK] {feat} ({time.time() - t0:.1f}s, VRAM peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GB)", flush=True)
    torch.cuda.reset_peak_memory_stats()

shutil.rmtree(WORK / ".preflight", ignore_errors=True)
print("PREFLIGHT PASSED", flush=True)
""", encoding="utf-8")

import tempfile

with step("Preflight: chạy thử Qwen-VL, SigLIP 2, OCR, ASR (tải sẵn model)"):
    # repo tìm weights VietOCR ở <tempdir>/<model>.pth trước khi tải từ mạng
    _vw_local = Path(tempfile.gettempdir()) / f"{load_cfg()['features']['ocr'].get('pretrained_model') or 'vgg_transformer'}.pth"
    if OCR_SOURCE != "tesseract" and VIETOCR_WEIGHTS_CACHE and Path(VIETOCR_WEIGHTS_CACHE).exists() and not _vw_local.exists():
        shutil.copy(VIETOCR_WEIGHTS_CACHE, _vw_local)
        log(f"Dùng weights VietOCR từ cache Drive: {VIETOCR_WEIGHTS_CACHE}")
    run_cmd([sys.executable, str(PREFLIGHT_PY), ",".join(f for f, _ in FEATURE_ORDER)], cwd=WORKSPACE)
    if OCR_SOURCE != "tesseract" and VIETOCR_WEIGHTS_CACHE and _vw_local.exists() and not Path(VIETOCR_WEIGHTS_CACHE).exists():
        Path(VIETOCR_WEIGHTS_CACHE).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(_vw_local, VIETOCR_WEIGHTS_CACHE)
        log(f"Đã lưu weights VietOCR lên Drive: {VIETOCR_WEIGHTS_CACHE}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 9 — TẢI (aria2c -x16 -s16) & GIẢI NÉN VIDEO
#  • Đủ đĩa → tải song song 2 file rồi giải nén lần lượt; thiếu đĩa → tải/giải nén từng batch.
#  • Chỉ giải nén *.mp4 (bỏ __MACOSX/._*), làm phẳng vào RAW_VIDEO_DIR, đuôi chuẩn hoá thành .mp4.
#  • Xoá .zip ngay sau khi giải nén. Ghi manifest (video nào thuộc batch nào) để đối soát/nén.
# ════════════════════════════════════════════════════════════════════════════
import urllib.request, zipfile

MANIFEST_PATH = Path(WORKSPACE) / "batch_manifest.json"


def load_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {"batches": {}}


def save_manifest(m):
    MANIFEST_PATH.write_text(json.dumps(m, indent=2, ensure_ascii=False))


def remote_size(url):
    try:
        with urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=30) as r:
            if r.headers.get("Content-Length"):
                return int(r.headers["Content-Length"])
    except Exception:
        pass
    try:
        req = urllib.request.Request(url, headers={"Range": "bytes=0-0"})
        with urllib.request.urlopen(req, timeout=30) as r:
            cr = r.headers.get("Content-Range", "")
            if "/" in cr and cr.split("/")[-1].isdigit():
                return int(cr.split("/")[-1])
    except Exception:
        pass
    return None


def zip_path(batch):
    return Path(DOWNLOAD_DIR) / Path(BATCHES[batch]).name


def aria2_download(batches):
    Path(DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)
    lst = Path(DOWNLOAD_DIR) / "aria2_input.txt"
    lst.write_text("".join(f"{BATCHES[b]}\n  out={zip_path(b).name}\n" for b in batches))
    run_cmd(["aria2c", "-x", "16", "-s", "16", "-k", "1M", "-j", str(len(batches)),
             "--file-allocation=none", "--continue=true", "--auto-file-renaming=false",
             "--max-tries=20", "--retry-wait=10", "--summary-interval=30",
             "--show-console-readout=false", "--console-log-level=warn",
             "-d", DOWNLOAD_DIR, "-i", str(lst)],
            heartbeat=lambda: " | ".join(f"{b}: {fmt_bytes(zip_path(b).stat().st_size if zip_path(b).exists() else 0)}"
                                         for b in batches))
    for b in batches:
        zp = zip_path(b)
        if not zp.exists():
            raise RuntimeError(f"Không tải được {zp.name}")
        exp = SIZES.get(b)
        if exp and zp.stat().st_size != exp:
            raise RuntimeError(f"{zp.name}: kích thước {zp.stat().st_size} != {exp} (tải chưa đủ)")
        log(f"   {zp.name}: {fmt_bytes(zp.stat().st_size)}")


def extract_batch(batch):
    zp = zip_path(batch)
    with zipfile.ZipFile(zp) as zf:
        entries = [i for i in zf.infolist() if not i.is_dir()]
    vids = [i for i in entries if i.filename.lower().endswith(".mp4")
            and "__MACOSX" not in i.filename and not Path(i.filename).name.startswith("._")]
    need = sum(i.file_size for i in vids)
    log(f"   {zp.name}: {len(entries)} file, {len(vids)} video mp4 ({fmt_bytes(need)} sau giải nén)")
    others = [i.filename for i in entries if i not in vids]
    if others:
        log(f"   bỏ qua {len(others)} file không phải video, ví dụ: {others[:5]}")
    if not vids:
        raise RuntimeError(f"{zp.name} không chứa file .mp4 nào")
    if disk_free() < need + 5 * 2**30:
        raise RuntimeError(f"Không đủ đĩa để giải nén {zp.name}: cần {fmt_bytes(need)}, còn {fmt_bytes(disk_free())}")

    tmp = Path(RAW_VIDEO_DIR) / f"_extract_{batch}"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    run_cmd(["unzip", "-q", "-o", "-C", str(zp), "*.mp4", "-d", str(tmp)])   # file rác ._* được lọc ở dưới

    stems = []
    for p in sorted(tmp.rglob("*")):
        if not p.is_file() or p.suffix.lower() != ".mp4" or p.name.startswith("._"):
            continue
        dest = Path(RAW_VIDEO_DIR) / f"{p.stem}.mp4"
        if dest.exists() or (Path(WORKSPACE) / "data" / "videos" / dest.name).exists():
            raise RuntimeError(f"Trùng video_id '{p.stem}' giữa các batch/thư mục con")
        shutil.move(str(p), dest)
        stems.append(p.stem)
    shutil.rmtree(tmp, ignore_errors=True)
    if len(stems) != len(vids):
        raise RuntimeError(f"{batch}: giải nén được {len(stems)}/{len(vids)} video")
    zp.unlink()                                   # xoá zip ngay để giải phóng đĩa
    log(f"   {batch}: {len(stems)} video → {RAW_VIDEO_DIR} (đã xoá {zp.name})")

    m = load_manifest()
    m["batches"][batch] = {"url": BATCHES[batch], "zip_bytes": SIZES.get(batch),
                           "video_bytes": need, "videos": sorted(stems), "status": "extracted"}
    save_manifest(m)


Path(RAW_VIDEO_DIR).mkdir(parents=True, exist_ok=True)
PENDING = [b for b in BATCHES if b not in load_manifest()["batches"]]
SIZES = {b: remote_size(BATCHES[b]) for b in BATCHES}
for b in BATCHES:
    log(f"{b}: {BATCHES[b]} → {fmt_bytes(SIZES[b]) if SIZES[b] else 'không rõ kích thước'}"
        + ("" if b in PENDING else "  (đã giải nén trước đó → bỏ qua)"))

if PENDING:
    margin = DISK_SAFETY_MARGIN_GB * 2**30
    known = all(SIZES.get(b) for b in PENDING)
    total = sum(SIZES[b] for b in PENDING) if known else None
    # song song: zip1 + zip2 + video(batch đang giải nén) ~ ≤ 2 × tổng
    PARALLEL = known and disk_free() >= 2 * total + margin
    log(f"Đĩa trống {fmt_bytes(disk_free())} → chế độ {'SONG SONG' if PARALLEL else 'TUẦN TỰ (tiết kiệm đĩa)'}")
    if known and disk_free() < 1.1 * total + margin:
        log("⚠️  Đĩa có thể không đủ cho toàn bộ video + keyframe. Cân nhắc runtime có đĩa lớn hơn.")
    if PARALLEL:
        with step(f"Tải song song {', '.join(PENDING)} (aria2c)"):
            aria2_download(PENDING)
        for b in PENDING:
            with step(f"Giải nén {b} & xoá zip"):
                extract_batch(b)
    else:
        for b in PENDING:
            with step(f"Tải {b} (aria2c)"):
                aria2_download([b])
            with step(f"Giải nén {b} & xoá zip"):
                extract_batch(b)

MANIFEST = load_manifest()
ALL_VIDEOS = sorted(v for b in MANIFEST["batches"].values() for v in b["videos"])
_counts = ", ".join(f"{b}={len(x['videos'])}" for b, x in MANIFEST["batches"].items())
log(f"Tổng: {len(ALL_VIDEOS)} video ({_counts})")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 10 — aic51-cli add -d -m -k -a : keyframes + thumbnails + audio (.wav) + video_info
#  -m (--move): chuyển video từ RAW_VIDEO_DIR vào data/videos thay vì copy → không nhân đôi dung lượng,
#  RAW_VIDEO_DIR trống ngay khi add xong. data/videos giữ tới khi ASR xong (ASR đọc FPS từ đó).
# ════════════════════════════════════════════════════════════════════════════
DATA = Path(WORKSPACE) / "data"
FEAT_DIR = Path(WORKSPACE) / "features"


def list_stems(d, ext=".jpg"):
    d = Path(d)
    if not d.is_dir():
        return set()
    return {e.name[: -len(ext)] for e in os.scandir(d) if e.is_file() and e.name.endswith(ext) and not e.name.startswith(".")}


def count_keyframes():
    kf = DATA / "keyframes"
    n = sum(len(list_stems(kf / v)) for v in ALL_VIDEOS) if kf.is_dir() else 0
    done = sum(1 for v in ALL_VIDEOS if (kf / v).is_dir())
    return f"{done}/{len(ALL_VIDEOS)} video có thư mục keyframe, {n} keyframe"


def has_audio_stream(video):
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "a:0", "-show_entries",
                        "stream=codec_name", "-of", "csv=p=0", str(video)], capture_output=True, text=True)
    return bool(r.stdout.strip())


with step("aic51-cli add: keyframes + thumbnails + audio + video_info"):
    raw = sorted(Path(RAW_VIDEO_DIR).glob("*.mp4"))
    log(f"{len(raw)} video trong {RAW_VIDEO_DIR} cần add ({fmt_bytes(sum(p.stat().st_size for p in raw))})")
    if raw:
        run_cmd(["aic51-cli", "add", RAW_VIDEO_DIR, "-d", "-m", "-k", "-a"], cwd=WORKSPACE, heartbeat=count_keyframes)

    problems, silent, n_kf = [], [], 0
    known_silent = set(load_manifest().get("silent_videos", []))
    for v in ALL_VIDEOS:
        vid_file = DATA / "videos" / f"{v}.mp4"
        kf, th = list_stems(DATA / "keyframes" / v), list_stems(DATA / "thumbnails" / v)
        n_kf += len(kf)
        if not vid_file.exists() and not (FEAT_DIR / v).is_dir():   # (data/videos bị xoá hợp lệ sau ASR)
            problems.append(f"{v}: thiếu data/videos/{v}.mp4")
        if not kf:
            problems.append(f"{v}: không có keyframe")
        elif kf != th:
            problems.append(f"{v}: keyframes({len(kf)}) != thumbnails({len(th)})")
        if not (DATA / "video_info" / f"{v}.json").exists():
            problems.append(f"{v}: thiếu video_info json")
        if not (DATA / "audio" / f"{v}.wav").exists():
            if v in known_silent or (vid_file.exists() and not has_audio_stream(vid_file)):
                silent.append(v)
            else:
                problems.append(f"{v}: thiếu data/audio/{v}.wav dù video có track audio")
    leftover = sorted(Path(RAW_VIDEO_DIR).glob("*.mp4"))
    if leftover:
        problems.append(f"{len(leftover)} video chưa được add (vẫn nằm trong {RAW_VIDEO_DIR}): {[p.name for p in leftover[:5]]}")

    m = load_manifest()
    m["silent_videos"] = sorted(silent)
    for b in m["batches"].values():
        b["status"] = "added"
    save_manifest(m)
    SILENT_VIDEOS = set(silent)
    log(f"{len(ALL_VIDEOS)} video | {n_kf} keyframe | {len(ALL_VIDEOS) - len(silent)} audio .wav | "
        f"{len(silent)} video không có track audio {sorted(silent)[:10]}")
    log(f"Dung lượng: keyframes {fmt_bytes(dir_size(DATA / 'keyframes'))}, thumbnails {fmt_bytes(dir_size(DATA / 'thumbnails'))}, "
        f"audio {fmt_bytes(dir_size(DATA / 'audio'))}, videos {fmt_bytes(dir_size(DATA / 'videos'))}")
    if problems:
        for p in problems[:50]:
            log("   ❌ " + p)
        raise RuntimeError(f"add chưa hoàn chỉnh: {len(problems)} lỗi (xem log ở trên)")
    shutil.rmtree(RAW_VIDEO_DIR, ignore_errors=True)   # video thô đã được move → thư mục trống
    log(f"Đã dọn {RAW_VIDEO_DIR}.")

EXPECTED = {v: sorted(list_stems(DATA / "keyframes" / v)) for v in ALL_VIDEOS}
TOTAL_FRAMES = sum(len(x) for x in EXPECTED.values())

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 11 — HÀM CHẠY 1 FEATURE: aic51-cli analyse --use-xxx --keep-going
#  Sau mỗi lần chạy: đếm frame còn thiếu .npy; nếu còn thiếu → giảm 1/2 batch_size và chạy lại
#  (CLI tự bỏ qua frame đã có, nên chạy lại chỉ xử lý phần thiếu).
# ════════════════════════════════════════════════════════════════════════════
def missing_frames(feature, exclude=()):
    miss = {}
    for v, frames in EXPECTED.items():
        if v in exclude:
            continue
        lack = [f for f in frames if not (FEAT_DIR / v / f / f"{feature}.npy").exists()]
        if lack:
            miss[v] = lack
    return miss


def feature_progress(feature):
    done = sum(1 for v, fr in EXPECTED.items() for f in fr if (FEAT_DIR / v / f / f"{feature}.npy").exists())
    return f"{feature}: {done}/{TOTAL_FRAMES} frame ({100 * done / max(TOTAL_FRAMES, 1):.1f}%)"


def run_feature(feature, flag):
    fail_json = Path(WORKSPACE) / "provenance" / "analysis" / f"{feature}.failures.json"
    exclude = SILENT_VIDEOS if feature == "asr" else set()
    extra = {"OMP_THREAD_LIMIT": "1"} if (feature == "ocr" and OCR_SOURCE == "tesseract") else {}
    with step(f"Trích xuất feature: {feature}  ({flag})"):
        if exclude:
            log(f"   {len(exclude)} video không có audio sẽ bị ASR báo lỗi (bình thường) và được điền rỗng sau.")
        for attempt in range(1, MAX_FEATURE_ATTEMPTS + 1):
            fail_json.unlink(missing_ok=True)
            log(f"   Lần chạy {attempt}/{MAX_FEATURE_ATTEMPTS} — {feature_progress(feature)}")
            run_cmd(["aic51-cli", "analyse", flag, "--keep-going"], cwd=WORKSPACE,
                    env=build_env(extra), heartbeat=lambda: feature_progress(feature))
            if fail_json.exists():
                fails = json.loads(fail_json.read_text()).get("failures", [])
                real = [f for f in fails if f["video_id"] not in exclude]
                log(f"   CLI ghi nhận {len(fails)} video lỗi ({len(real)} ngoài nhóm không-audio)")
                for f in real[:10]:
                    log(f"      - {f['video_id']}: {f['error'][:300]}")
            miss = missing_frames(feature, exclude)
            n_miss = sum(len(x) for x in miss.values())
            log(f"   Sau lần {attempt}: {feature_progress(feature)} | còn thiếu {n_miss} frame / {len(miss)} video")
            if not miss:
                break
            if attempt < MAX_FEATURE_ATTEMPTS:
                cfg = load_cfg()
                bs = cfg["features"][feature].setdefault("analyse", {}).get("batch_size") or 1
                if bs > 1:
                    cfg["features"][feature]["analyse"]["batch_size"] = max(1, bs // 2)
                    save_cfg(cfg)
                    log(f"   ↻ Giảm batch_size {bs} → {max(1, bs // 2)} và chạy lại phần thiếu")
        else:
            log(f"   ⚠️  {feature}: vẫn thiếu frame sau {MAX_FEATURE_ATTEMPTS} lần — bước đối soát sẽ báo chi tiết.")


FEATURE_FLAGS = dict(FEATURE_ORDER)
log(f"{len(ALL_VIDEOS)} video, {TOTAL_FRAMES} keyframe cần trích xuất cho {len(FEATURE_ORDER)} feature.")

In [ ]:
# CELL 12 — Feature 1/4: Qwen3-VL-Embedding-2B (qwen_vl.npy, 2048 chiều)
run_feature("qwen_vl", FEATURE_FLAGS["qwen_vl"])

In [ ]:
# CELL 13 — Feature 2/4: SigLIP 2 ViT-SO400M-14-378 (image_siglip2_so400m-378.npy, 1152 chiều)
run_feature("image_siglip2_so400m-378", FEATURE_FLAGS["image_siglip2_so400m-378"])

In [ ]:
# CELL 14 — Feature 3/4: OCR PaddleOCR(detect) + VietOCR(recognize) (ocr.npy — chuỗi text đã chuẩn hoá)
run_feature("ocr", FEATURE_FLAGS["ocr"])

In [ ]:
# CELL 15 — Feature 4/4: ASR WhisperX large-v3-turbo (asr.npy — câu thoại gần timestamp keyframe)
run_feature("asr", FEATURE_FLAGS["asr"])

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 16 — HẬU XỬ LÝ ASR: điền asr.npy rỗng cho video không có audio, xoá data/videos
# ════════════════════════════════════════════════════════════════════════════
import numpy as np

with step("Hậu xử lý ASR & dọn data/videos", heavy=False):
    filled = 0
    if FILL_EMPTY_ASR_FOR_SILENT_VIDEOS:
        for v in sorted(SILENT_VIDEOS):
            for f in EXPECTED[v]:
                p = FEAT_DIR / v / f / "asr.npy"
                if not p.exists():
                    p.parent.mkdir(parents=True, exist_ok=True)
                    np.save(p, np.array(""))   # cùng định dạng khi ASR không khớp câu nào: chuỗi rỗng
                    filled += 1
        log(f"Điền asr.npy rỗng cho {filled} frame của {len(SILENT_VIDEOS)} video không có audio.")
        if SILENT_VIDEOS:
            m = load_manifest(); m["asr_filled_empty_videos"] = sorted(SILENT_VIDEOS); save_manifest(m)

    asr_missing = missing_frames("asr")
    if DELETE_VIDEOS_AFTER_ASR and not asr_missing:
        freed = dir_size(DATA / "videos")
        shutil.rmtree(DATA / "videos", ignore_errors=True)
        log(f"Đã xoá data/videos (giải phóng {fmt_bytes(freed)}).")
    elif asr_missing:
        log(f"Giữ data/videos vì ASR còn thiếu {sum(len(x) for x in asr_missing.values())} frame (cần để chạy lại).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 17 — ĐỐI SOÁT TOÀN VẸN
#  1) Mọi frame của M09/M10 có đủ 4/4 file .npy (không rỗng).
#  2) data/thumbnails/<video>/*.jpg khớp 1-1 với features/<video>/<frame>/ (và với keyframes).
#  3) Kiểm tra mẫu: vector đúng số chiều, đã chuẩn hoá; OCR/ASR là chuỗi.
# ════════════════════════════════════════════════════════════════════════════
REQUIRED_FILES = [f"{f}.npy" for f, _ in FEATURE_ORDER]
cfg_now = load_cfg()
DIMS = {f: (cfg_now["features"][f].get("index") or {}).get("dim") for f, _ in FEATURE_ORDER}

with step("Đối soát tính toàn vẹn M09 + M10"):
    report = {"generated_at": now_str(), "repo_commit": globals().get("REPO_COMMIT"),
              "required_files": REQUIRED_FILES, "batches": {}, "errors": []}
    errors = report["errors"]
    video_batch = {v: b for b, x in MANIFEST["batches"].items() for v in x["videos"]}
    per_batch = {b: {"videos": 0, "frames": 0, "complete_frames": 0} for b in MANIFEST["batches"]}
    rng = np.random.default_rng(0)

    for v in ALL_VIDEOS:
        b = video_batch[v]
        thumbs = list_stems(DATA / "thumbnails" / v)
        fdir = FEAT_DIR / v
        frames = {e.name for e in os.scandir(fdir) if e.is_dir()} if fdir.is_dir() else set()
        per_batch[b]["videos"] += 1
        per_batch[b]["frames"] += len(thumbs)
        if thumbs != frames:
            errors.append({"video": v, "type": "thumbnail_feature_mismatch",
                           "only_in_thumbnails": sorted(thumbs - frames)[:20],
                           "only_in_features": sorted(frames - thumbs)[:20],
                           "n_thumbs": len(thumbs), "n_feature_frames": len(frames)})
        if set(EXPECTED[v]) != thumbs:
            errors.append({"video": v, "type": "keyframe_thumbnail_mismatch"})
        incomplete = []
        for f in sorted(frames):
            lack = [r for r in REQUIRED_FILES
                    if not (fdir / f / r).exists() or (fdir / f / r).stat().st_size == 0]
            if lack:
                incomplete.append((f, lack))
            else:
                per_batch[b]["complete_frames"] += 1
        if incomplete:
            errors.append({"video": v, "type": "missing_npy", "n_frames": len(incomplete),
                           "examples": [{"frame": f, "missing": l} for f, l in incomplete[:10]]})
        # kiểm tra mẫu 2 frame/video
        ok_frames = sorted(set(frames) - {f for f, _ in incomplete})
        for f in (rng.choice(ok_frames, size=min(2, len(ok_frames)), replace=False) if ok_frames else []):
            for feat, _ in FEATURE_ORDER:
                try:
                    a = np.load(fdir / f / f"{feat}.npy", allow_pickle=False)
                    if DIMS.get(feat):
                        assert a.shape == (DIMS[feat],), f"shape {a.shape} != ({DIMS[feat]},)"
                        assert np.isfinite(a).all() and abs(float(np.linalg.norm(a)) - 1) < 2e-2, "norm != 1"
                    else:
                        assert a.dtype.kind == "U", f"dtype {a.dtype} không phải chuỗi"
                except Exception as e:
                    errors.append({"video": v, "type": "bad_npy", "frame": str(f), "feature": feat, "error": str(e)})

    report["batches"] = per_batch
    report["n_videos"] = len(ALL_VIDEOS)
    report["n_frames"] = sum(x["frames"] for x in per_batch.values())
    report["silent_videos_asr_empty"] = sorted(SILENT_VIDEOS)
    REPORT_PATH = Path(WORKSPACE) / "verification_report.json"
    REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False))

    for b, x in per_batch.items():
        log(f"   {b}: {x['videos']} video | {x['frames']} frame | đủ 4/4 file: {x['complete_frames']}/{x['frames']}")
    if errors:
        by_type = {}
        for e in errors:
            by_type[e["type"]] = by_type.get(e["type"], 0) + 1
        log(f"   ❌ {len(errors)} lỗi đối soát: {by_type} — chi tiết: {REPORT_PATH}")
        for e in errors[:15]:
            log(f"      {json.dumps(e, ensure_ascii=False)[:400]}")
        if STRICT_VERIFY:
            raise RuntimeError("Đối soát thất bại → dừng trước khi nén/upload (đặt STRICT_VERIFY=False để bỏ qua).")
    else:
        log(f"   ✅ PASS: {report['n_videos']} video, {report['n_frames']} frame — đủ 4/4 file, "
            f"thumbnails ↔ features khớp 1-1.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 18 — NÉN 2 FILE ZIP & UPLOAD LÊN GOOGLE DRIVE
#  features_M09_M10.zip   → features/<video>/<frame>/*.npy
#  thumbnails_M09_M10.zip → data/thumbnails/<video>/*.jpg
#  (đường dẫn trong zip tính từ gốc workspace → giải nén thẳng vào workspace khác là dùng được)
#  KHÔNG nén data/keyframes. ZIP được tạo trên SSD local rồi copy lên Drive (nhanh & ổn định hơn ghi trực tiếp).
# ════════════════════════════════════════════════════════════════════════════
import zipfile

Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
FEATURES_ZIP = Path(EXPORT_DIR) / FEATURES_ZIP_NAME
THUMBS_ZIP = Path(EXPORT_DIR) / THUMBNAILS_ZIP_NAME

need = dir_size(FEAT_DIR) + dir_size(DATA / "thumbnails")
if DELETE_KEYFRAMES_BEFORE_ZIP or disk_free() < 1.2 * need + 5 * 2**30:
    freed = dir_size(DATA / "keyframes")
    shutil.rmtree(DATA / "keyframes", ignore_errors=True)
    log(f"Đã xoá data/keyframes để có chỗ nén (giải phóng {fmt_bytes(freed)}).")


def make_zip(out, rel_dirs, level, n_expected, ext):
    out.unlink(missing_ok=True)
    lst = Path(EXPORT_DIR) / f"{out.stem}.dirs.txt"
    lst.write_text("\n".join(rel_dirs) + "\n")
    # -r -@ : đọc danh sách thư mục từ stdin (tránh dòng lệnh quá dài)
    run_cmd(f"zip -r -q -{level} {shlex.quote(str(out))} -@ < {shlex.quote(str(lst))}", cwd=WORKSPACE,
            heartbeat=lambda: f"{out.name}: {fmt_bytes(out.stat().st_size) if out.exists() else '...'}")
    lst.unlink()
    run_cmd(["unzip", "-tqq", str(out)])                       # kiểm tra CRC toàn bộ file zip
    with zipfile.ZipFile(out) as zf:
        n = sum(1 for n in zf.namelist() if n.endswith(ext))
    log(f"   {out.name}: {fmt_bytes(out.stat().st_size)}, {n} file {ext} (kỳ vọng {n_expected})")
    if n != n_expected:
        raise RuntimeError(f"{out.name}: số file {n} != {n_expected}")


n_frames = sum(len(x) for x in EXPECTED.values())
with step(f"Nén {FEATURES_ZIP_NAME}"):
    make_zip(FEATURES_ZIP, [f"features/{v}" for v in ALL_VIDEOS], 1, n_frames * len(FEATURE_ORDER), ".npy")
with step(f"Nén {THUMBNAILS_ZIP_NAME}"):
    make_zip(THUMBS_ZIP, [f"data/thumbnails/{v}" for v in ALL_VIDEOS], 0, n_frames, ".jpg")   # JPEG: lưu, không nén lại


def copy_with_progress(src, dst, chunk=64 * 2**20):
    total, done, t0, last = src.stat().st_size, 0, time.time(), 0.0
    tmp = dst.with_name(dst.name + ".part")
    with open(src, "rb") as fi, open(tmp, "wb") as fo:
        while True:
            buf = fi.read(chunk)
            if not buf:
                break
            fo.write(buf)
            done += len(buf)
            if time.time() - last >= 30:
                log(f"   {dst.name}: {fmt_bytes(done)}/{fmt_bytes(total)} "
                    f"({100 * done / max(total, 1):.0f}%, {done / 2**20 / max(time.time() - t0, 1e-6):.0f} MB/s)")
                last = time.time()
        fo.flush()
        os.fsync(fo.fileno())
    tmp.replace(dst)
    if dst.stat().st_size != total:
        raise RuntimeError(f"Copy lỗi: {dst} có {dst.stat().st_size} bytes, kỳ vọng {total}")
    log(f"   ✅ {dst} ({fmt_bytes(total)})")


with step("Upload 2 file ZIP lên Google Drive"):
    Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    for z in (FEATURES_ZIP, THUMBS_ZIP):
        copy_with_progress(z, Path(DRIVE_OUTPUT_DIR) / z.name)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 19 — TỔNG KẾT, COPY LOG LÊN DRIVE, FLUSH DRIVE
# ════════════════════════════════════════════════════════════════════════════
log("=" * 100)
log("TỔNG KẾT THỜI GIAN CÁC BƯỚC")
for r in STEP_RESULTS:
    log(f"   {'✅' if r['ok'] else '❌'} {r['step'][:70]:70s} {fmt_elapsed(r['seconds']):>18s}  [{r['start']} → {r['end']}]")
log(f"TỔNG THỜI GIAN PIPELINE: {fmt_elapsed(time.time() - PIPELINE_START)}")
log(f"Kết quả trên Drive: {Path(DRIVE_OUTPUT_DIR) / FEATURES_ZIP_NAME}")
log(f"                    {Path(DRIVE_OUTPUT_DIR) / THUMBNAILS_ZIP_NAME}")
sys_stats("KẾT THÚC")

if COPY_LOGS_TO_DRIVE:
    logs_dst = Path(DRIVE_OUTPUT_DIR) / "logs"
    logs_dst.mkdir(parents=True, exist_ok=True)
    for f in (LOG_FILE, Path(WORKSPACE) / "verification_report.json", MANIFEST_PATH, CONFIG_PATH):
        if Path(f).exists():
            shutil.copy(f, logs_dst / Path(f).name)
    (logs_dst / "step_timings.json").write_text(json.dumps(STEP_RESULTS, indent=2, ensure_ascii=False))
    log(f"Đã copy log/báo cáo vào {logs_dst}")

if FLUSH_DRIVE_AT_END:
    from google.colab import drive
    print("Đang flush Google Drive (chờ upload xong)...", flush=True)
    drive.flush_and_unmount()
    print("✅ Drive đã flush & unmount. Hoàn tất pipeline M09 + M10.", flush=True)